# Week 01 · Day 05 — Hugging Face Ecosystem

**Topics:** Hub navigation · `pipeline()` · Inference API · Spaces


In [ ]:
%pip install -q transformers huggingface_hub torch

In [ ]:
import os
from huggingface_hub import HfApi

# Optional: set token for private models and higher rate limits
# os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")

api = HfApi()

## 1 · Exploring the Hub

In [ ]:
# List top models for a task
models = api.list_models(
    task="text-classification",
    sort="downloads",
    limit=5,
)

print("Top text-classification models by downloads:")
for m in models:
    print(f"  {m.id:<50} {m.downloads:>10,} downloads")

In [ ]:
# Search by keyword
sentiment_models = api.list_models(
    search="sentiment",
    task="text-classification",
    sort="likes",
    limit=5,
)

print("Top sentiment models by likes:")
for m in sentiment_models:
    print(f"  {m.id:<50} {getattr(m, 'likes', 0):>6} likes")

## 2 · The `pipeline()` API

`pipeline()` is a high-level API that handles tokenization, model loading, and post-processing. One line to run inference.

In [ ]:
from transformers import pipeline

# Sentiment analysis — downloads model on first run (~67MB)
sentiment = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    # device=0 to use GPU; device=-1 (default) for CPU
)

texts = [
    "I absolutely love this product!",
    "The service was terrible and I waited for an hour.",
    "It's okay — nothing special.",
]

results = sentiment(texts)
for text, result in zip(texts, results):
    print(f"{result['label']:10s} ({result['score']:.3f}) — {text}")

In [ ]:
# Named Entity Recognition
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

text = "Elon Musk founded SpaceX in 2002 in Hawthorne, California."
entities = ner(text)

print(f"Text: {text}")
print("Entities:")
for ent in entities:
    print(f"  {ent['entity_group']:<5} ({ent['score']:.3f}) — '{ent['word']}'")

In [ ]:
# Summarization
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

long_text = """
The Hugging Face Hub is a platform with over 350,000 models, 75,000 datasets, and 150,000 
demo apps. Researchers and practitioners can share and discover state-of-the-art ML models 
across modalities including text, image, audio, and video. The Hub provides Git-based version 
control for models, built-in dataset viewers, model cards for documentation, and seamless 
integration with the transformers library. Organizations can host models privately with 
fine-grained access control, making it suitable for both open-source and enterprise use cases.
""".strip()

summary = summarizer(long_text, max_length=80, min_length=30, do_sample=False)
print(f"Summary: {summary[0]['summary_text']}")

## 3 · Hugging Face Inference API

The Inference API lets you call models hosted on Hugging Face servers — no local GPU required. Free tier with rate limits; PRO tier for higher throughput.

In [ ]:
from huggingface_hub import InferenceClient

# Requires HF_TOKEN for most models
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    hf_client = InferenceClient(token=hf_token)

    # Text generation via Inference API
    response = hf_client.text_generation(
        prompt="Explain gradient descent in one paragraph:",
        model="mistralai/Mistral-7B-Instruct-v0.3",
        max_new_tokens=150,
        temperature=0.3,
    )
    print(response)
else:
    print("HF_TOKEN not set — set it to call the Inference API")
    print("Pattern:")
    print("  hf_client = InferenceClient(token=os.getenv('HF_TOKEN'))")
    print("  response = hf_client.text_generation(prompt=..., model=..., max_new_tokens=150)")

In [ ]:
# Chat completion format (for instruction-tuned models)
if hf_token:
    hf_client = InferenceClient(token=hf_token)
    messages = [
        {"role": "user", "content": "What is the difference between RAG and fine-tuning?"}
    ]
    response = hf_client.chat_completion(
        messages=messages,
        model="mistralai/Mistral-7B-Instruct-v0.3",
        max_tokens=200,
    )
    print(response.choices[0].message.content)
else:
    print("Set HF_TOKEN to run chat completion")

## 4 · Loading a Model and Tokenizer Directly

Use `AutoModel` and `AutoTokenizer` when you need custom inference logic beyond what `pipeline()` provides.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

text = "This movie was surprisingly good!"
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

with torch.no_grad():
    logits = model(**inputs).logits

probs = torch.softmax(logits, dim=-1)
label_id = probs.argmax().item()
label = model.config.id2label[label_id]

print(f"Text:  {text}")
print(f"Label: {label}")
print(f"Probs: {dict(zip(model.config.id2label.values(), probs[0].tolist()))}")

## 5 · Model Cards and Metadata

In [ ]:
from huggingface_hub import model_info

info = model_info("distilbert-base-uncased-finetuned-sst-2-english")
print(f"Model: {info.id}")
print(f"Task:  {info.pipeline_tag}")
print(f"Downloads: {info.downloads:,}")
print(f"Likes:     {info.likes}")
if info.tags:
    print(f"Tags:  {info.tags[:5]}")

## Summary

- `pipeline(task, model=...)` — one line to run inference; handles everything.
- `AutoTokenizer` + `AutoModel` — full control when you need custom logic.
- Inference API — call hosted models without a local GPU; needs `HF_TOKEN`.
- Hub has 350k+ models; filter by task, sort by downloads to find proven options.
- Always read the model card before deploying — check training data, limitations, license.

**Next:** [Local LLMs](week01-day05-local-llms.ipynb)